In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


In [2]:
import os
import torch
from torchvision import transforms
import timm # provides pretrained models
from timm.layers import SwiGLUPacked


timm_kwargs = {
            'img_size': 224, 
            'patch_size': 14, 
            'depth': 24,
            'num_heads': 24,
            'init_values': 1e-5, 
            'embed_dim': 1536,
            'mlp_ratio': 2.66667*2,
            'num_classes': 0, 
            'no_embed_class': True,
            'mlp_layer': timm.layers.SwiGLUPacked, 
            'act_layer': torch.nn.SiLU, 
            'reg_tokens': 8, 
            'dynamic_img_size': True
        }
model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
model.to('mps')
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1536, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1536, out_features=4608, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=1536, out_features=1536, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
      (mlp): GluMlp(
        (fc1): Linear(in_features=1536, out_features=8192, bias=True)
        (act): SiLU()
        (drop1): Dropout(p=0.0, inplace=False)
    

In [3]:
torch.cuda.is_available() 

False

In [4]:
torch.mps.is_available()

True

In [2]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModel
from timm.data.transforms_factory import create_transform
from timm.data import resolve_data_config
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tqdm import tqdm
import numpy as np

In [ ]:


# # -------------------------------
# # Configuration
# # -------------------------------
DATA_DIR = "/Users/jk/Documents/Lab2/visium/python_analysis/cpath/visium_patches/55um"
MODEL_NAME = "MahmoodLab/UNI2-h"   # Hugging Face repo
BATCH_SIZE = 32 # number of images processed in parallel
DEVICE = "mps" 

# # -------------------------------
# # Load model & processor
# # -------------------------------
# processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
# # model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
# # model.eval()  # Freeze encoder

# # -------------------------------
# # Image preprocessing
# # -------------------------------
# transform = transforms.Compose(
#     [
#         transforms.Resize(224),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ]
# )
transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))

# -------------------------------
# Create datasets
# -------------------------------
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Extract class names
class_names = dataset.classes
print("Classes:", class_names)

# -------------------------------
# Extract embeddings
# -------------------------------
features, labels = [], []
with torch.no_grad(): # disable gradient computation
    for images, lbls in tqdm(loader, desc="Extracting UNI-2 embeddings"):
        images = images.to('mps')
        outputs = model(images)  # directly use your timm model
        # UNI uses CLS token output
        if isinstance(outputs, (list, tuple)):
            embeddings = outputs[0]
        else:
            embeddings = outputs
        features.append(embeddings.cpu().numpy())
        labels.append(lbls.numpy())

features = np.concatenate(features)
labels = np.concatenate(labels)



Classes: ['cluster0', 'cluster1', 'cluster2', 'cluster3', 'cluster4', 'cluster5']


Extracting UNI-2 embeddings: 100%|██████████| 3140/3140 [1:33:02<00:00,  1.78s/it]


In [8]:
features

array([[-0.0983479 , -0.42170322, -0.16660303, ...,  0.32553568,
         0.26622114,  0.3254263 ],
       [ 0.27279627,  0.13655457, -0.13894445, ..., -0.13388641,
        -0.48153216, -0.2049264 ],
       [ 0.15675083, -0.21041402, -0.18347281, ..., -0.18868996,
         0.3742479 ,  0.22215225],
       ...,
       [ 0.32962784, -0.19950134, -0.38573954, ..., -0.04130968,
         0.5568426 , -0.5247867 ],
       [-0.00565244,  0.04363465, -0.10068324, ..., -0.26047975,
         0.29833823,  0.03779237],
       [ 0.16135082,  0.18441445, -0.09654977, ...,  0.20608021,
         0.35263735,  0.54862154]], shape=(100472, 1536), dtype=float32)

In [9]:
labels

array([0, 0, 0, ..., 5, 5, 5], shape=(100472,))

In [ ]:
np.save("features/features_uni2_55um.npy", features)
np.save("labels/labels_uni2_55um.npy", labels)

In [ ]:
import numpy as np
features = np.load("features/features_uni2_55um.npy")
labels = np.load("labels/labels_uni2_55um.npy")

In [4]:
# -------------------------------
# Split train/validation
# -------------------------------
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, stratify=labels, random_state=42)


In [9]:
# -------------------------------
# Train simple classifier
# -------------------------------
clf = LogisticRegression(max_iter=500, multi_class='multinomial')
clf.fit(X_train, y_train)

# -------------------------------
# Evaluate
# -------------------------------
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred))

/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.49      0.49      0.49      3941
           1       0.45      0.45      0.45      3900
           2       0.52      0.60      0.55      3782
           3       0.42      0.32      0.36      2763
           4       0.51      0.47      0.49      2868
           5       0.54      0.58      0.56      2841

    accuracy                           0.49     20095
   macro avg       0.49      0.48      0.48     20095
weighted avg       0.48      0.49      0.49     20095



/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Train on all data
clf_final = LogisticRegression(max_iter=500, multi_class='multinomial')
clf_final.fit(features, labels)

# Save for later
import joblib
joblib.dump(clf_final, "logistic_classifier/niche_classifier_final_uni2_55um.joblib")

/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


['niche_classifier_final_uni2_55um.joblib']